In [ ]:
import torch
from torch import nn
import gymnasium as gym
import ale_py
import cv2
from matplotlib import pyplot as plt
from collections import deque
import random
import copy

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
memory_capacity = 30000
num_episodes = 100000
epsilon = lambda frames: max(0.1, 1.0 - frames / (2 * 1e5))
max_episode_steps = 10000
learning_rate = 1e-4
minibatch_size = 64
discount_factor = 0.99
target_update_freq = 2000

In [ ]:
gym.register_envs(ale_py)
env = gym.make("ALE/Breakout-v5", render_mode=None, max_episode_steps=max_episode_steps)

In [ ]:
def preprocess_image(image, crop_top=20):
    image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) / 255.0
    image = cv2.resize(image, (84, 110))
    return image[crop_top:crop_top+84, :]

In [ ]:
obs, info = env.reset()
print(obs.shape)

plt.figure(figsize=(10,5))

plt.subplot(1, 2, 1)
plt.imshow(obs)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(preprocess_image(obs), cmap='gray')
plt.axis('off')

plt.show()
preprocess_image(obs)

In [ ]:
class DQN(nn.Module):
    def __init__(self, num_actions=4): # inp shape: (b, 4, 84, 84)
        super().__init__()

        self.layers = nn.Sequential(
            nn.Conv2d(4, 16, kernel_size=8, stride=4),  # 84x84 - 20x20
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=4, stride=2),  # 20x20 - 9x9
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * 9 * 9, 256),
            nn.ReLU(),
            nn.Linear(256, num_actions)
        )
        
    def forward(self, x):
        return self.layers(x)

In [ ]:
model = DQN().to(device)
target_model = copy.deepcopy(model).to(device) # for stability

print(model)

In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
replay_memory = deque(maxlen=memory_capacity)

In [ ]:
total_frames = 0

model.train()
target_model.eval()


for episode in range(num_episodes):
    episode_reward = 0
    episode_frames = 0

    obs, info = env.reset()
    frame_stack = deque([preprocess_image(obs)] * 4, maxlen=4)

    done = False

    while not done:
        if random.random() < epsilon(total_frames):
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                inp = torch.tensor(frame_stack, dtype=torch.float32).unsqueeze(0).to(device)  # add batch dim
                action = torch.argmax(model(inp), dim=1).item()

        obs, reward, terminated, truncated, info = env.step(action)
        reward -= 0.01  # general time penalty

        episode_reward += reward
        episode_frames += 1
        total_frames += 1
        done = terminated or truncated

        old_frame_stack = torch.tensor(frame_stack, dtype=torch.float32)
        frame_stack.append(preprocess_image(obs))
        replay_memory.append((old_frame_stack, action, reward, torch.tensor(frame_stack, dtype=torch.float32), done))

        if len(replay_memory) >= minibatch_size and total_frames % 4 == 0:
            batch = random.sample(replay_memory, minibatch_size)
            old_frame_stack, actions, rewards, new_frame_stack, dones = zip(*batch)

            old_frame_stack = torch.stack(old_frame_stack).to(device)
            actions = torch.tensor(actions, dtype=torch.long).to(device)
            rewards = torch.tensor(rewards, dtype=torch.float32).to(device)
            new_frame_stack = torch.stack(new_frame_stack).to(device)
            dones = torch.tensor(dones, dtype=torch.float32).to(device)

            with torch.no_grad():
                targets = (
                    rewards
                    + (1 - dones)
                    * discount_factor
                    * torch.max(target_model(new_frame_stack), dim=1).values
                )
            
            preds = model(old_frame_stack).gather(1, actions.unsqueeze(1)).squeeze(1)

            loss = loss_fn(preds, targets)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
            if total_frames % target_update_freq == 0:
                target_model.load_state_dict(model.state_dict())
                print(f"Updated target network at frame {total_frames}")
            
    if episode % 10 == 0:
        torch.save(model.state_dict(), f"models/dqn_model_episode_{episode}.pth")

    print(f"Episode {episode + 1}: Total Reward: {episode_reward:.3f}  Average Reward: {episode_reward / (episode_frames + 1):.7f} Frames: {episode_frames} Epsilon: {epsilon(total_frames):.3f}")